# Payments Qualifier Agent — MVP Prototype

Runs the full four-pillar qualification pipeline (Reach → Regulatory → Sanctions → Fraud) against synthetic TCH RTP payment instructions, using a real compiled LangGraph orchestrator with input/output guardrails and a hand-rolled observability layer.

See the repo README for full architecture, scope, and disclosures. This notebook is the runnable entry point only — all actual logic lives in `src/qualifier/`.

**Optional:** set `ANTHROPIC_API_KEY` as a Colab secret (or environment variable) to enable a real LLM call for the Sanctions near-miss Tree-of-Thought branch. Without it, near-miss cases still escalate correctly, just without live model reasoning (see the graceful-degradation note in `orchestrator.py`).

In [ ]:
# If running in Colab: upload the whole repo folder first, or clone from GitHub, then adjust this path.
import sys, os
sys.path.insert(0, "../src")

!pip install -q langgraph anthropic

In [ ]:
import json
from pathlib import Path

DATA_DIR = Path("../data")

participants = json.load(open(DATA_DIR / "participants.json"))
chunk_store = json.load(open(DATA_DIR / "regulatory_chunks.json"))
watchlist = json.load(open(DATA_DIR / "ofac_watchlist.json"))
transaction_log = json.load(open(DATA_DIR / "transaction_log.json"))
sample_payments = json.load(open(DATA_DIR / "sample_payments.json"))["scenarios"]
expected_outcomes = {e["scenario_id"]: e for e in json.load(open(DATA_DIR / "expected_outcomes.json"))["expected_outcomes"]}

print(f"Loaded {len(sample_payments)} sample scenarios")

## Set up the LLM client (optional)

If `ANTHROPIC_API_KEY` is set, this uses a real client for the Sanctions ToT branch. Otherwise falls back to `None`, and near-miss cases will still escalate correctly (fail-safe), just without live model reasoning.

In [ ]:
import os

llm_client = None
if os.environ.get("ANTHROPIC_API_KEY"):
    import anthropic
    llm_client = anthropic.Anthropic()
    print("Using a real Anthropic client for the Sanctions ToT branch.")
else:
    print("No ANTHROPIC_API_KEY found -- Sanctions near-miss cases will escalate without live ToT reasoning.")
    print("Set the key as a Colab secret / environment variable to enable it.")

## Run a single scenario

In [ ]:
from qualifier.agents.orchestrator import qualify_payment

scenario = next(s for s in sample_payments if s["scenario_id"] == "SCN-01-HAPPY-PATH")
result = qualify_payment(
    scenario["instruction"], participants, chunk_store, watchlist, transaction_log,
    llm_client=llm_client,
)

print("Final verdict:", result["final_verdict"])
print("Escalated:", result["escalated"], "| Role:", result["escalation_role"])
for p in result["pillar_results"]:
    print(f"  {p['pillar']:12s} {p['verdict']:6s} {p['reason']}")

## Run the full evaluation suite

This regenerates `eval/results.csv`, `eval/results_summary.md`, and `eval/trace_log.jsonl` from scratch, checked against the hand-computed ground truth in `data/expected_outcomes.json`.

In [ ]:
import subprocess
result = subprocess.run(["python", "run_eval.py"], cwd="../eval", capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## View results

In [ ]:
import csv
with open("../eval/results.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        match = "PASS" if row["matches_expected"] == "True" else "MISMATCH"
        print(f"{row['scenario_id']:38s} {row['final_verdict']:26s} [{match}]")

## Try the interactive demos

Open `examples/interactive_demo.html` in a browser to explore all 8 scenarios visually (embeds real captured tool output, no server needed).

Open `examples/tot_live_demo.html` (in claude.ai, or self-hosted with your own API key) to see a genuinely live-generated ToT run for the Sanctions near-miss scenario.